# TP 2 : ingérer du JSON depuis une API, les communes de France

**Durée : 25 minutes, en binôme.**

L'[API Géo](https://geo.api.gouv.fr), publiée par l'État, renvoie en JSON les communes, leur population, leurs codes
postaux et les coordonnées de leur centre. Elle est gratuite et sans clé. On s'en sert pour enrichir les accidents
du TP 1.

À la fin du TP, vous saurez :
- appeler une API proprement : paramètres, `timeout`, code HTTP, réponse brute sauvegardée ;
- aplatir un JSON imbriqué avec `pd.json_normalize` ;
- enchaîner des appels, et reprendre là où l'on s'était arrêté ;
- vérifier une jointure, et remplacer une boucle par un calcul vectorisé.

Comme au TP 1, une cellule `assert` suit chaque exercice, et un exercice non rempli s'arrête sur « Exercice à
compléter ».

In [ ]:
import json
import time
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import requests

API = "https://geo.api.gouv.fr/communes"
CLEAN = Path("data/clean")

# Sans réseau, remplacez par JOUR = "2026-10-25" : c'est la copie de secours fournie
JOUR = date.today().isoformat()
RAW_API = Path("data/raw/api_geo") / JOUR


# Appelée par les cellules d'exercice tant que vous ne les avez pas remplies
def a_completer():
    raise NotImplementedError("Exercice à compléter : remplacez a_completer() par votre code")

Les accidents viennent du TP 1. S'il n'est pas terminé, la cellule les relit depuis le CSV.

In [ ]:
if (CLEAN / "accidents" / "caract.parquet").exists():
    caract = pd.read_parquet(CLEAN / "accidents" / "caract.parquet")
else:
    caract = pd.read_csv(
        "data/raw/accidents_2024/caract-2024.csv",
        sep=";",
        decimal=",",
        dtype={"Num_Acc": "str", "dep": "str", "com": "str"},
    )
print(len(caract), "accidents,", caract["dep"].nunique(), "départements")

## Partie 1 : un appel (5 min)

Appelez l'API pour les communes de Corse-du-Sud (`codeDepartement=2A`), avec les champs
`nom,code,population,centre,codesPostaux`. Passez les paramètres par `params` plutôt que dans l'URL, ajoutez un
`timeout`, et levez une erreur si le code HTTP n'est pas 200.

In [ ]:
reponse = a_completer()  # À compléter
communes_2a = reponse.json()
print(reponse.status_code, reponse.headers["content-type"], len(communes_2a), "communes")
print(json.dumps(communes_2a[1], ensure_ascii=False, indent=2))

In [ ]:
assert isinstance(communes_2a, list) and len(communes_2a) > 100
assert {"nom", "code", "population", "centre", "codesPostaux"} <= set(communes_2a[0])

**Règle :** on sauvegarde la réponse brute avant de la transformer. Si la transformation a un défaut, on la corrige et
on la relance sans rappeler l'API. Sauvegardez `communes_2a` dans `RAW_API / "essai_2A.json"`.

In [ ]:
a_completer()  # À compléter

In [ ]:
assert json.loads((RAW_API / "essai_2A.json").read_text(encoding="utf-8")) == communes_2a

## Partie 2 : aplatir le JSON (6 min)

`pd.json_normalize` transforme les objets imbriqués en colonnes séparées par un point. Regardez le résultat, puis
remplacez la colonne `centre.coordinates`, une liste `[longitude, latitude]`, par deux colonnes `lon` et `lat`.
Supprimez les colonnes `centre.*`.

In [ ]:
pd.json_normalize(communes_2a).head(3)

In [ ]:
def aplatir_communes(records: list[dict]) -> pd.DataFrame:
    a_completer()  # À compléter


communes_2a_df = aplatir_communes(communes_2a)
communes_2a_df.head(3)

In [ ]:
assert {"lon", "lat"} <= set(communes_2a_df.columns)
assert not any(c.startswith("centre") for c in communes_2a_df.columns)
assert communes_2a_df["lat"].between(41, 43).all(), "la Corse-du-Sud est entre 41 et 43 degrés de latitude"

Une commune peut avoir plusieurs codes postaux. Avec `record_path="codesPostaux"` et `meta=["code", "nom"]`,
`json_normalize` produit une ligne par code postal. Construisez cette table et renommez la colonne `0` en
`code_postal`.

In [ ]:
codes_postaux = a_completer()  # À compléter
codes_postaux.head()

In [ ]:
assert len(codes_postaux) == sum(len(c["codesPostaux"]) for c in communes_2a)
assert list(codes_postaux.columns) == ["code_postal", "code", "nom"]

## Partie 3 : tous les départements des accidents (6 min)

Écrivez `extraire(departements, dossier)` :
- une seule `requests.Session` pour tous les appels : la connexion est réutilisée, c'est plus rapide ;
- pour chaque département, le fichier brut `dossier / f"communes_{dep}.json"` ;
- **si le fichier existe déjà, on ne rappelle pas l'API** : relancer après une coupure reprend où l'on s'était arrêté ;
- une pause de 0,1 seconde entre deux appels, par politesse envers le service.

On demande aussi les arrondissements municipaux (`type`) : la partie 4 montre pourquoi. Le premier passage prend
de 20 à 40 secondes selon le réseau. Le second, presque rien.

In [ ]:
CHAMPS = "nom,code,codeDepartement,population,centre,codesPostaux,type"
TYPES = "commune-actuelle,arrondissement-municipal"
departements = sorted(caract["dep"].unique())

In [ ]:
def extraire(departements: list[str], dossier: Path, pause: float = 0.1) -> list[Path]:
    a_completer()  # À compléter


debut = time.perf_counter()
paths = extraire(departements, RAW_API)
print(f"{len(paths)} fichiers en {time.perf_counter() - debut:.1f} s")

Relancez la cellule précédente : le temps doit tomber sous la seconde. Construisez ensuite **une seule fois** le
DataFrame de toutes les communes : une liste de tous les enregistrements, puis `aplatir_communes`. Ajouter les lignes
une à une dans une boucle recopierait le tableau à chaque tour. Renommez `codeDepartement` en `dep` et `codesPostaux`
en `codes_postaux`, et passez `population` en `Int64`.

In [ ]:
records = a_completer()  # À compléter
communes = a_completer()  # À compléter
communes.head()

In [ ]:
assert len(communes) > 34_000
assert {"code", "nom", "dep", "population", "type", "lon", "lat"} <= set(communes.columns)
assert str(communes["population"].dtype) == "Int64"

## Partie 4 : vérifier la jointure (4 min)

Avant d'enrichir les accidents, on mesure quelle part d'entre eux retrouve sa commune. Calculez ce taux d'abord avec
les seules communes (`type == "commune-actuelle"`), puis avec toutes les lignes. Pour le premier calcul, affichez les
cinq départements qui concentrent les accidents sans correspondance.

In [ ]:
taux_tout = a_completer()  # À compléter : les deux taux, puis les départements

In [ ]:
assert taux_tout > 0.998

**Question.** Que s'est-il passé avec les communes seules ? Que faire des accidents encore sans commune ?

*Votre réponse :*

## Partie 5 : accidents pour 10 000 habitants (2 min)

Calculez, par département, le nombre d'accidents pour 10 000 habitants, et affichez les cinq premiers. Attention à
la population : Paris figure à la fois comme commune et comme somme de ses arrondissements.

In [ ]:
population = a_completer()  # À compléter
taux = a_completer()  # À compléter
taux.head()

In [ ]:
assert population["75"] < 2_500_000, "Paris est-il compté deux fois ?"

## Partie 6 : remplacer une boucle par un calcul vectorisé (7 min)

Quelle distance sépare chaque accident du centre de sa commune ? La fonction `haversine` est fournie. Elle accepte des
nombres comme des tableaux NumPy. On compare trois façons de l'appliquer :
1. `iterrows`, fournie ;
2. `apply` ligne par ligne, à compléter ;
3. un seul appel sur les colonnes entières, à compléter.

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))


acc = caract.merge(communes[["code", "lat", "lon"]], left_on="com", right_on="code", suffixes=("", "_centre"))


def avec_iterrows(df):
    return [haversine(r["lat"], r["long"], r["lat_centre"], r["lon"]) for _, r in df.iterrows()]

In [ ]:
def avec_apply(df):
    a_completer()  # À compléter


def vectorise(df):
    a_completer()  # À compléter

In [ ]:
temps = {}
for fonction in [avec_iterrows, avec_apply, vectorise]:
    debut = time.perf_counter()
    distances = np.asarray(fonction(acc), dtype=float)
    temps[fonction.__name__] = time.perf_counter() - debut
    assert np.allclose(distances, vectorise(acc))
print({nom: f"{t:.4f} s" for nom, t in temps.items()})
print(f"Vectorisé {temps['avec_iterrows'] / temps['vectorise']:.0f} fois plus rapide que iterrows")

In [ ]:
acc["distance_km"] = vectorise(acc)
acc["distance_km"].describe().round(1)

**Question.** Certains accidents sont à plus de 50 km du centre de leur commune. Faut-il corriger ces lignes dans le
script d'ingestion ?

*Votre réponse :*

## Écrire le résultat

In [ ]:
(CLEAN / "api_geo").mkdir(parents=True, exist_ok=True)
communes.to_parquet(CLEAN / "api_geo" / "communes.parquet", index=False)